<div align="center">

# SHIV'S LLM ENGINEERING JOURNEY

### DAY 04 · Tokens, Memory & Conversations

*Exploring tokenization, conversation messages, and follow-up interactions with Gemini.*

**Learn · Build · Experiment · Share**

</div>

---

> **Learning in public** — a hands-on notebook documenting my progress in Generative AI and Agentic AI.

### What you'll explore

- Tokenization with tiktoken
- Conversation memory concepts
- Message roles and structure
- Gemini API calls

---

**Author:** Shiv Bhadouria  
**Focus:** Generative AI · LLM APIs · Prompt Engineering · Agentic AI



# Day 4

## Tokenizing with code

In [8]:
import tiktoken

# gpt-6-luna isn't in tiktoken yet, so we use the encoding of recent OpenAI models
encoding = tiktoken.get_encoding("o200k_base")

tokens = encoding.encode("Hi my name is Shiv and I love AI currently I am learning about LLMs and how to use them effectively. I am also exploring the different types of LLMs available in the market and their capabilities.")

In [9]:
tokens

[12194,
 922,
 1308,
 382,
 150937,
 326,
 357,
 3047,
 20837,
 6821,
 357,
 939,
 7524,
 1078,
 451,
 19641,
 82,
 326,
 1495,
 316,
 1199,
 1373,
 18130,
 13,
 357,
 939,
 1217,
 29457,
 290,
 2647,
 6009,
 328,
 451,
 19641,
 82,
 2839,
 306,
 290,
 2910,
 326,
 1043,
 23736,
 13]

In [10]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
150937 =  Shiv
326 =  and
357 =  I
3047 =  love
20837 =  AI
6821 =  currently
357 =  I
939 =  am
7524 =  learning
1078 =  about
451 =  L
19641 = LM
82 = s
326 =  and
1495 =  how
316 =  to
1199 =  use
1373 =  them
18130 =  effectively
13 = .
357 =  I
939 =  am
1217 =  also
29457 =  exploring
290 =  the
2647 =  different
6009 =  types
328 =  of
451 =  L
19641 = LM
82 = s
2839 =  available
306 =  in
290 =  the
2910 =  market
326 =  and
1043 =  their
23736 =  capabilities
13 = .


In [11]:
encoding.decode([7524])

' learning'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [19]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No Google API key found. Check your .env file.")

else:
    print("Google API key found!")

    gemini = OpenAI(
        base_url=GEMINI_BASE_URL,
        api_key=google_api_key
    )

Google API key found!


### Gemini-compatible explanation
I'm configuring the OpenAI Python Client library to communicate with Google's Gemini API through its OpenAI-compatible endpoint at https://generativelanguage.googleapis.com/v1beta/openai/. This allows me to interact with the Gemini 3.1 Flash Lite model using a Google API key, without running any LLM locally on my computer.

In [20]:
from openai import OpenAI

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=google_api_key
)

### A message to GEMINI is a list of dicts

In [24]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Shiv. Can you tell me about the Gemini 3.1 model?"},
    ]

In [25]:
response = gemini.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

response.choices[0].message.content

'Hi Shiv! It’s nice to meet you.\n\nTo give you the most accurate answer: **As of right now, there is no official "Gemini 3.1" model.**\n\nThe current state-of-the-art lineup from Google is the **Gemini 1.5 series**, which includes **Gemini 1.5 Pro** and **Gemini 1.5 Flash**.\n\nIt is possible you might be confusing the naming conventions with other models in the industry (like Llama 3.1 from Meta, which was released recently). \n\n**Here is a quick breakdown to clear up the confusion:**\n\n1.  **Llama 3.1:** This is the model series released by Meta (Facebook) in July 2024. It includes their most powerful open-weights model to date, Llama 3.1 405B. People are talking about this one quite a bit right now.\n2.  **Gemini 1.5:** Google’s current flagship models. They are best known for their massive "context window" (the ability to process up to 2 million tokens at once, meaning you can upload entire books, long codebases, or hour-long videos for the model to analyze).\n\nIf you are inter

### OK let's now ask a follow-up question

In [26]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [27]:
response = gemini.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

response.choices[0].message.content

"I don’t know your name! As an AI, I don’t have access to your personal information or identity unless you’ve shared it with me in this specific conversation. \n\nIf you'd like, you can tell me your name, and I’ll be happy to remember it for the duration of our chat!"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [28]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Shiv bhadouria!"},
    {"role": "assistant", "content": "Hi Shiv bhadouria! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [30]:
response = gemini.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

response.choices[0].message.content

'Your name is Shiv bhadouria!'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Shiv!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

